# 03 - Feature Engineering & Representation

> **Project:** AI-Based Interview Preparation Assistant  
> **Course:** AI252TA – Machine Learning Operations (Semester V)  
> **Status:** Phase 1 – Completed  

### Objective & Purpose
Inspect the fitted feature pipeline, engineered feature distributions, and correlations with target.

---


In [1]:
import joblib
import yaml
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'params.yaml').exists())
params = yaml.safe_load((PROJECT_ROOT / 'params.yaml').read_text())

# Import custom classes so joblib can unpickle the feature pipeline
import sys
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from features.build_features import TextLengthFeatures, ReferenceSimilarityFeatures  # noqa: F401

print(f'Project root: {PROJECT_ROOT}')

Project root: /home/likithshashidhar/MLOps


## Load Feature Pipeline

In [2]:
pipeline_path = PROJECT_ROOT / 'models/feature_pipeline.joblib'
feature_pipeline = joblib.load(pipeline_path)

print(f'Pipeline type: {type(feature_pipeline).__name__}')
print(f'Feature names (first 20): {feature_pipeline.get_feature_names_out()[:20]}')
print(f'Total features: {len(feature_pipeline.get_feature_names_out())}')

Pipeline type: ColumnTransformer
Feature names (first 20): ['tfidf__1st' 'tfidf__20' 'tfidf__2nd' 'tfidf__2nd element'
 'tfidf__ability' 'tfidf__ability access' 'tfidf__able' 'tfidf__able grow'
 'tfidf__abstract' 'tfidf__abstract data' 'tfidf__abstraction'
 'tfidf__access' 'tfidf__access element' 'tfidf__access elements'
 'tfidf__access entire' 'tfidf__access linked' 'tfidf__access list'
 'tfidf__access modify' 'tfidf__access specific'
 'tfidf__access specifiers']
Total features: 2824


In [3]:
# Inspect pipeline structure
print('Pipeline transformers:')
for name, trans, cols in feature_pipeline.transformers_:
    print(f'  {name}: {type(trans).__name__}, columns={cols}')

Pipeline transformers:
  tfidf: TfidfVectorizer, columns=candidate_answer
  length_features: TextLengthFeatures, columns=['candidate_answer']
  ref_sim_features: ReferenceSimilarityFeatures, columns=['candidate_answer', 'reference_answer']
  remainder: str, columns=['question_id', 'question', 'score_grader1', 'score_grader2', 'score_avg', 'quality_label']


## Transform Train Data & Inspect Feature Distributions

In [4]:
# Load train data
train_df = pd.read_parquet(PROJECT_ROOT / 'data/processed/train.parquet')
X_train = train_df.drop(columns=[params['data']['target_column']])
y_train = train_df[params['data']['target_column']].values

# Transform
X_train_feat = feature_pipeline.transform(X_train)

print(f'Transformed shape: {X_train_feat.shape}')
print(f'Feature type: {type(X_train_feat)}')
print(f'Sparsity: {1 - X_train_feat.nnz / (X_train_feat.shape[0] * X_train_feat.shape[1]):.2%}')

Transformed shape: (1561, 2824)
Feature type: <class 'scipy.sparse._csr.csr_matrix'>
Sparsity: 99.31%


In [5]:
# Get feature names and identify engineered feature indices
feature_names = feature_pipeline.get_feature_names_out()

tfidf_mask = np.array([n.startswith('tfidf__') for n in feature_names])
length_mask = np.array([n.startswith('length_features__') for n in feature_names])
ref_sim_mask = np.array(['cosine_sim' in n for n in feature_names])

print(f'TF-IDF features: {tfidf_mask.sum()}')
print(f'Length features: {length_mask.sum()}')
print(f'Reference similarity features: {ref_sim_mask.sum()}')

length_feature_names = feature_names[length_mask]
ref_sim_feature_name = feature_names[ref_sim_mask][0] if ref_sim_mask.any() else None
print(f'Length feature names: {length_feature_names}')
print(f'Reference similarity feature: {ref_sim_feature_name}')

TF-IDF features: 2818
Length features: 5
Reference similarity features: 1
Length feature names: ['length_features__char_count' 'length_features__token_count'
 'length_features__sentence_count' 'length_features__unique_token_ratio'
 'length_features__mean_word_length']
Reference similarity feature: ref_sim_features__candidate_reference_cosine_sim


In [6]:
# Convert to dense for engineered features (they're small)
if hasattr(X_train_feat, 'toarray'):
    X_dense = X_train_feat.toarray()
else:
    X_dense = X_train_feat

# Extract engineered features
X_length = X_dense[:, length_mask]
X_ref_sim = X_dense[:, ref_sim_mask].flatten() if ref_sim_mask.any() else None

# Create DataFrame for easier plotting
length_df = pd.DataFrame(X_length, columns=length_feature_names)
length_df['quality_label'] = y_train
length_df['class_name'] = length_df['quality_label'].map(params['data']['target_classes'])

if X_ref_sim is not None:
    ref_sim_df = pd.DataFrame({
        'cosine_similarity': X_ref_sim,
        'quality_label': y_train,
        'class_name': pd.Series(y_train).map(params['data']['target_classes']).values
    })
else:
    ref_sim_df = None

## Engineered Feature Distributions per Class (Train)

In [7]:
# Length feature distributions by class
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, feat in enumerate(length_feature_names):
    ax = axes[i]
    for cls in range(4):
        subset = length_df[length_df['quality_label'] == cls][feat]
        ax.hist(subset, bins=30, alpha=0.5, label=params['data']['target_classes'][cls], density=True)
    ax.set_title(feat)
    ax.legend()
    ax.set_xlabel('Value')
    ax.set_ylabel('Density')

# Hide empty subplot
axes[-1].axis('off')
plt.tight_layout()
plt.show()

/tmp/ipykernel_67902/2299946362.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# Reference similarity distribution by class
if ref_sim_df is not None:
    fig, ax = plt.subplots(figsize=(10, 6))
    for cls in range(4):
        subset = ref_sim_df[ref_sim_df['quality_label'] == cls]['cosine_similarity']
        ax.hist(subset, bins=30, alpha=0.5, label=params['data']['target_classes'][cls], density=True)
    ax.set_title('Candidate-Reference Cosine Similarity by Class')
    ax.set_xlabel('Cosine Similarity')
    ax.set_ylabel('Density')
    ax.legend()
    plt.tight_layout()
    plt.show()

/tmp/ipykernel_67902/1284131309.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
# Summary statistics per class
print('=== Length Features: Mean per Class ===')
length_means = length_df.groupby('class_name')[length_feature_names].mean().round(3)
print(length_means.to_string())

if ref_sim_df is not None:
    print('\n=== Cosine Similarity: Mean per Class ===')
    ref_means = ref_sim_df.groupby('class_name')['cosine_similarity'].mean().round(4)
    print(ref_means.to_string())

=== Length Features: Mean per Class ===
                   length_features__char_count  length_features__token_count  length_features__sentence_count  length_features__unique_token_ratio  length_features__mean_word_length
class_name                                                                                                                                                                           
Excellent                              121.685                        21.995                            2.015                                0.861                              4.712
Good                                   123.195                        22.032                            2.032                                0.870                              4.803
Needs Improvement                      119.313                        21.305                            2.082                                0.867                              4.743
Poor                                    60.177    

> **Takeaway:** Higher quality answers tend to be longer (more characters, tokens, sentences) and have higher cosine similarity with reference answers. The 'Poor' class shows low similarity and short length, while 'Excellent' answers are long and semantically close to references.

## Correlation with score_avg (Train Only)

In [10]:
# Load score_avg from train
score_avg = train_df['score_avg'].values

# Correlation of length features with score_avg
length_corrs = {}
for feat in length_feature_names:
    corr = np.corrcoef(length_df[feat], score_avg)[0, 1]
    length_corrs[feat] = corr

print('=== Length Features vs score_avg Correlation ===')
for feat, corr in sorted(length_corrs.items(), key=lambda x: abs(x[1]), reverse=True):
    print(f'  {feat}: {corr:.4f}')

=== Length Features vs score_avg Correlation ===
  length_features__unique_token_ratio: -0.1065
  length_features__token_count: 0.0909
  length_features__char_count: 0.0886
  length_features__sentence_count: 0.0517
  length_features__mean_word_length: -0.0189


In [11]:
# Correlation of cosine similarity with score_avg
if ref_sim_df is not None:
    ref_corr = np.corrcoef(ref_sim_df['cosine_similarity'], score_avg)[0, 1]
    print(f'Cosine similarity vs score_avg: {ref_corr:.4f}')

Cosine similarity vs score_avg: 0.3332


In [12]:
# Correlation heatmap
corr_df = length_df[list(length_feature_names) + ['quality_label']].copy()
corr_df['score_avg'] = score_avg
if ref_sim_df is not None:
    corr_df['cosine_similarity'] = X_ref_sim

corr_matrix = corr_df.corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Feature Correlations (Train)')
plt.tight_layout()
plt.show()

/tmp/ipykernel_67902/382472991.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


> **Takeaway:** Answer length features show very weak correlation with score_avg (|r| <= 0.11); cosine similarity has moderate correlation (r ~ 0.33). Length is a weak predictor; cosine similarity is a moderate predictor of answer quality.

## Feature Count Summary

In [13]:
print('=== Feature Count Summary ===')
print(f'TF-IDF features: {tfidf_mask.sum()}')
print(f'Engineered length features: {length_mask.sum()}')
print(f'Reference similarity feature: {int(ref_sim_mask.sum())}')
print(f'Total features: {len(feature_names)}')

# Sparsity
total_elements = X_train_feat.shape[0] * X_train_feat.shape[1]
nnz = X_train_feat.nnz if hasattr(X_train_feat, 'nnz') else np.count_nonzero(X_dense)
print(f'Non-zero elements: {nnz:,} / {total_elements:,} ({nnz/total_elements:.2%})')

=== Feature Count Summary ===
TF-IDF features: 2818
Engineered length features: 5
Reference similarity feature: 1
Total features: 2824
Non-zero elements: 30,422 / 4,408,264 (0.69%)
